In [29]:
import pandas as pd
import numpy as np
df = pd.read_csv('data/raw/day_approach_maskedID_timeseries.csv')
df = df.replace(-0.01, np.nan)

feature_cols = ['total km', 'nr. sessions', 'km Z3-4', 'km Z5-T1-T2',
                'km sprinting', 'strength training', 'hours alternative',
                'perceived exertion', 'perceived trainingSuccess',
                'perceived recovery']
feature_stats = []
for feature in feature_cols:
    cols = [feature]
    cols += [f"{feature}.{i}" for i in range(1, 7)]
    feature_stats.append( df[cols].agg(['max', 'mean', 'std', 'count'], axis=1).add_prefix(f"{feature}_") )
X = pd.concat(feature_stats, axis=1)

print(X.shape)
print(X.describe())

(42766, 40)
       total km_max  total km_mean  total km_std  total km_count  \
count  42766.000000   42766.000000  42766.000000         42766.0   
mean      13.853192       7.041486      5.039410             7.0   
std        8.486734       5.197299      2.859534             0.0   
min        0.000000       0.000000      0.000000             7.0   
25%        9.000000       3.242857      3.454035             7.0   
50%       13.400000       6.371429      5.126634             7.0   
75%       18.200000       9.985714      6.632352             7.0   
max       57.000000      31.471429     25.975492             7.0   

       nr. sessions_max  nr. sessions_mean  nr. sessions_std  \
count      42766.000000       42766.000000      42766.000000   
mean           1.321166           0.829264          0.447148   
std            0.569865           0.355146          0.214427   
min            0.000000           0.000000          0.000000   
25%            1.000000           0.714286          0.3

In [30]:
y = df['injury']
groups = df['Athlete ID']
print(y.shape)
print(groups.shape)
X.index.equals(y.index)
X.index.equals(groups.index)
y.sum()


(42766,)
(42766,)


np.int64(583)

In [31]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

athlete_df = (df.groupby("Athlete ID", as_index=False)["injury"].sum()
                .rename(columns={"injury": "total_injuries"}))

print("Injury count distribution:")
print(athlete_df["total_injuries"].describe().round(1), "\n")

MIN_PER_BIN = 7
injured = athlete_df["total_injuries"] > 0
n_uninjured = (~injured).sum()

athlete_df["injury_bin"] = "uninjured"
athlete_df.loc[injured, "injury_bin"] = pd.qcut(
    athlete_df.loc[injured, "total_injuries"],
    q=3, labels=["low", "mid", "high"], duplicates="drop"
).astype(str)

if n_uninjured < MIN_PER_BIN:
    athlete_df.loc[~injured, "injury_bin"] = "low"
    print(f"Only {n_uninjured} uninjured athletes: merged into 'low'")

print("Athletes per bin:")
print(athlete_df["injury_bin"].value_counts())
print("\nInjury-count range per bin:")
print(athlete_df.groupby("injury_bin")["total_injuries"].agg(["min", "max"]))

assert athlete_df["injury_bin"].value_counts().min() >= 2, "a bin is too small to stratify"

dev_athletes, holdout_athletes = train_test_split(
    athlete_df, test_size=0.15,
    stratify=athlete_df["injury_bin"], random_state=42)

is_holdout = groups.isin(holdout_athletes["Athlete ID"])

print("\nHoldout check:")
print(f"  athletes:  {len(holdout_athletes)} of {len(athlete_df)} "
      f"({len(holdout_athletes) / len(athlete_df):.1%})")
print(f"  rows:      {is_holdout.mean():.1%}")
print(f"  injuries:  {y[is_holdout].sum() / y.sum():.1%}")
print(f"  injury rate  holdout: {y[is_holdout].mean():.2%}   dev: {y[~is_holdout].mean():.2%}")
print("  bins:", holdout_athletes["injury_bin"].value_counts().to_dict())

Path("data/splits").mkdir(parents=True, exist_ok=True)
(holdout_athletes[["Athlete ID"]].sort_values("Athlete ID")
    .to_csv("data/splits/holdout_athlete_ids.csv", index=False))

X_dev, y_dev, groups_dev = X[~is_holdout], y[~is_holdout], groups[~is_holdout]
assert X_dev.index.equals(y_dev.index) and X_dev.index.equals(groups_dev.index)
print(f"\nDev set: {X_dev.shape[0]} rows, {groups_dev.nunique()} athletes, {y_dev.sum()} injuries")

Injury count distribution:
count    74.0
mean      7.9
std       8.3
min       0.0
25%       2.0
50%       5.0
75%      11.5
max      35.0
Name: total_injuries, dtype: float64 

Athletes per bin:
injury_bin
low          22
high         21
mid          20
uninjured    11
Name: count, dtype: int64

Injury-count range per bin:
            min  max
injury_bin          
high         10   35
low           1    4
mid           5    9
uninjured     0    0

Holdout check:
  athletes:  12 of 74 (16.2%)
  rows:      15.9%
  injuries:  15.8%
  injury rate  holdout: 1.35%   dev: 1.36%
  bins: {'low': 4, 'mid': 3, 'high': 3, 'uninjured': 2}

Dev set: 35973 rows, 62 athletes, 491 injuries


In [32]:
import json
from datetime import datetime, timezone
from sklearn.model_selection import StratifiedKFold

N_SPLITS = 5
SEEDS = [0, 1, 2, 3, 4]
HOLDOUT_PATH = "data/splits/holdout_athlete_ids.csv"
CV_SPLITS_PATH = "data/splits/cv_splits.json"

assert dev_athletes["injury_bin"].value_counts().min() >= N_SPLITS, (
    f"Smallest dev bin has fewer than {N_SPLITS} athletes; increase MIN_PER_BIN or lower N_SPLITS."
)

dev_ids = dev_athletes["Athlete ID"].to_numpy()
dev_bins = dev_athletes["injury_bin"].to_numpy()

splits_by_seed = {}
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    splits_by_seed[str(seed)] = [
        sorted(dev_ids[val_idx].tolist())
        for _, val_idx in skf.split(dev_ids, dev_bins)
    ]

payload = {
    "metadata": {
        "holdout_file": HOLDOUT_PATH,
        "id_col": "Athlete ID",
        "stratify_col": "injury_bin",
        "split_strategy": "StratifiedKFold (athlete-level)",
        "n_splits": N_SPLITS,
        "n_dev_athletes": int(len(dev_athletes)),
        "n_dev_rows": int(len(X_dev)),
        "created_at": datetime.now(timezone.utc).isoformat(),
    },
    "seeds": splits_by_seed,
}

with open(CV_SPLITS_PATH, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print(f"Saved {len(SEEDS)}x{N_SPLITS} CV splits to {CV_SPLITS_PATH}")


Saved 5x5 CV splits to data/splits/cv_splits.json


In [33]:
import time
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score


def evaluate(model, X, y, groups, cv, name=None, verbose=True):
    """Fit a fresh clone of `model` on every saved fold and score it.

    Returns one row per (seed, fold) with train/test AUC, test AP,
    the fold's injury rate, and AP lift (AP / injury rate).
    """
    name = name or type(model).__name__
    records = []
    t0 = time.time()

    for seed, folds in cv["seeds"].items():
        for k, test_ids in enumerate(folds):
            test_mask = groups.isin(test_ids).to_numpy()
            X_tr, y_tr = X[~test_mask], y[~test_mask]
            X_te, y_te = X[test_mask], y[test_mask]

            m = clone(model)                       # fresh, unfitted copy every fold
            m.fit(X_tr, y_tr)

            p_tr = m.predict_proba(X_tr)[:, 1]
            p_te = m.predict_proba(X_te)[:, 1]

            rate = y_te.mean()
            ap = average_precision_score(y_te, p_te)
            records.append({
                "model": name, "seed": int(seed), "fold": k,
                "train_auc": roc_auc_score(y_tr, p_tr),
                "test_auc": roc_auc_score(y_te, p_te),
                "test_ap": ap,
                "injury_rate": rate,
                "ap_lift": ap / rate,
            })

    results = pd.DataFrame(records)
    if verbose:
        s = summarize(results)
        print(f"{name}: test AUC {s['test_auc_mean']:.4f} ± {s['test_auc_std']:.4f} | "
              f"train AUC {s['train_auc_mean']:.4f} | "
              f"AP lift {s['ap_lift_mean']:.2f} ± {s['ap_lift_std']:.2f} | "
              f"{time.time() - t0:.0f}s")
    return results


def summarize(results):
    """Mean and std (ddof=1) of the main metrics across all folds."""
    out = {}
    for col in ["train_auc", "test_auc", "ap_lift"]:
        out[f"{col}_mean"] = results[col].mean()
        out[f"{col}_std"] = results[col].std()
    return out


def compare(results_a, results_b, metric="test_auc"):
    """Paired per-fold difference (b - a) on the same saved folds."""
    merged = results_a.merge(results_b, on=["seed", "fold"], suffixes=("_a", "_b"))
    diff = merged[f"{metric}_b"] - merged[f"{metric}_a"]
    print(f"{results_b['model'].iloc[0]} vs {results_a['model'].iloc[0]} ({metric}): "
          f"mean diff {diff.mean():+.4f}, b wins {int((diff > 0).sum())}/{len(diff)} folds")
    return diff

In [28]:
import xgboost as xgb
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

prevalence = DummyClassifier(strategy="prior")

logreg = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),   # fit on train fold only
    StandardScaler(),
    LogisticRegression(max_iter=2000),
)

xgb_base = xgb.XGBClassifier(
    max_depth=2, n_estimators=512, learning_rate=0.01,
    random_state=0, n_jobs=-1,
)

res_prev = evaluate(prevalence, X_dev, y_dev, groups_dev, cv, name="prevalence")
res_lr   = evaluate(logreg,     X_dev, y_dev, groups_dev, cv, name="logreg")
res_xgb  = evaluate(xgb_base,   X_dev, y_dev, groups_dev, cv, name="xgb_d2")

compare(res_lr, res_xgb)

prevalence: test AUC 0.5000 ± 0.0000 | train AUC 0.5000 | AP lift 1.00 ± 0.00 | 0s
logreg: test AUC 0.6173 ± 0.0317 | train AUC 0.6746 | AP lift 1.43 ± 0.22 | 1s
xgb_d2: test AUC 0.6156 ± 0.0280 | train AUC 0.7485 | AP lift 1.51 ± 0.37 | 9s
xgb_d2 vs logreg (test_auc): mean diff -0.0017, b wins 10/25 folds


0     0.019145
1    -0.006865
2     0.028940
3    -0.012425
4     0.014001
5    -0.032568
6     0.005048
7    -0.035395
8    -0.010147
9     0.034251
10   -0.015212
11    0.031913
12   -0.002890
13   -0.004531
14   -0.036914
15    0.025934
16    0.000154
17   -0.005157
18    0.027087
19   -0.021095
20   -0.035046
21   -0.009671
22   -0.006134
23    0.017476
24   -0.012153
dtype: float64

Average Precision: 0.0156
Test ROC AUC: 0.5707
Average Precision: 0.0171
Test ROC AUC: 0.5894
Average Precision: 0.0189
Test ROC AUC: 0.6254
Average Precision: 0.0214
Test ROC AUC: 0.6132
Average Precision: 0.0256
Test ROC AUC: 0.6555
Average Test ROC AUC: 0.6109 +/- 0.0327
Average Test AP: 0.0197 +/- 0.0039 


Train C Index: 0.6581756607524111
Test C Index: 0.5644504748982361

Train C Index: 0.657090555887627
Test C Index: 0.5621794871794872

Train C Index: 0.649598094671033
Test C Index: 0.6212319790301442

Train C Index: 0.6734845116348006
Test C Index: 0.5266579973992198

Train C Index: 0.6612867236891038
Test C Index: 0.6538952745849298

Test Average C Index: 0.5857 +/- 0.0510
